# Credit Card Fraud Detection – Decision Tree
**Member 2**  
**Algorithm:** Decision Tree Classifier  
**Dataset:** [Kaggle – Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

A Decision Tree recursively splits the feature space based on information gain (or Gini impurity). It is highly interpretable and can model non-linear boundaries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, f1_score
)

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Preprocessed Data

In [ ]:
X_train, X_test, y_train, y_test = joblib.load('preprocessed_data.pkl')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'max_depth': [4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

dt_base = DecisionTreeClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(dt_base, param_grid, cv=5,
                           scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
dt = grid_search.best_estimator_

## 3. Predictions & Evaluation

In [ ]:
y_pred = dt.predict(X_test)
y_prob = dt.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print(f'Avg Prec : {average_precision_score(y_test, y_prob):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')

## 4. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0],
            xticklabels=['Legitimate','Fraud'], yticklabels=['Legitimate','Fraud'])
axes[0].set_title('Confusion Matrix – Decision Tree')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='green', lw=2, label=f'AUC = {auc_score:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_title('ROC Curve – Decision Tree')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[2].plot(rec, prec, color='darkgreen', lw=2, label=f'AP = {ap:.4f}')
axes[2].set_title('Precision-Recall – Decision Tree')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].legend()

plt.tight_layout()
plt.savefig('dt_results.png', dpi=150)
plt.show()

## 5. Visualise the Tree (Truncated)

In [ ]:
plt.figure(figsize=(22, 8))
plot_tree(dt, max_depth=3, feature_names=list(X_test.columns),
          class_names=['Legit','Fraud'], filled=True, rounded=True,
          fontsize=9)
plt.title('Decision Tree (max depth shown: 3)')
plt.tight_layout()
plt.savefig('dt_tree.png', dpi=120)
plt.show()

## 6. Feature Importances

In [ ]:
fi_df = pd.DataFrame({'Feature': X_test.columns,
                       'Importance': dt.feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='Greens_r')
plt.title('Top 15 Feature Importances – Decision Tree')
plt.tight_layout()
plt.savefig('dt_feature_importance.png', dpi=150)
plt.show()

## 7. Summary
| Metric | Value |
|--------|-------|
| ROC-AUC | See output above |
| Average Precision | See output above |
| F1-Score (Fraud) | See output above |

**Observations:**
- GridSearchCV tuned `max_depth` and `criterion` to reduce overfitting.
- The tree is visualizable, making it easy to explain decisions.
- Limitation: prone to overfitting without pruning; generally weaker than ensembles.